In [ ]:
import ipaddress
import pandas as pd
import matplotlib.pyplot as plt

# Keep consistent with other analysis notebooks
TIME_OFFSET = 10800

In [ ]:
def plot_mean_regval_per_src_register(
    input_csv,
    src_ip,
    register_value,
    center_timestamp,
    interval_seconds,
    regval_col="modbus.regval_uint16",
    register_col="modbus.reference_num",
    time_offset_seconds=TIME_OFFSET,
):
    """
    1) Plot mean(modbus.regval_uint16) per (src IP, register) in 1-second bins for
    [center_timestamp - interval_seconds, center_timestamp + interval_seconds].
    """
    ipaddress.ip_address(src_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "frame.time_epoch", regval_col, register_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(epoch[epoch.notna()] + float(time_offset_seconds), unit="s", errors="coerce")

    df[regval_col] = pd.to_numeric(df[regval_col], errors="coerce")
    reg_num = pd.to_numeric(df[register_col], errors="coerce")

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    filtered = df[(df["ip.src"].astype(str) == src_ip) & (reg_num == float(register_value))].copy()
    window_df = filtered[(filtered["aligned_ts"] >= start_ts) & (filtered["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")[regval_col].mean().reindex(bins)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel(f"mean({regval_col})")
    plt.title(
        f"Mean register value for src={src_ip}, register={register_value} around {center_ts} (+/-{interval_seconds}s)"
    )
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


def plot_std_regval_per_src_register(
    input_csv,
    src_ip,
    register_value,
    center_timestamp,
    interval_seconds,
    regval_col="modbus.regval_uint16",
    register_col="modbus.reference_num",
    time_offset_seconds=TIME_OFFSET,
):
    """
    2) Plot std(modbus.regval_uint16) per (src IP, register) in 1-second bins.
    """
    ipaddress.ip_address(src_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "frame.time_epoch", regval_col, register_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(epoch[epoch.notna()] + float(time_offset_seconds), unit="s", errors="coerce")

    df[regval_col] = pd.to_numeric(df[regval_col], errors="coerce")
    reg_num = pd.to_numeric(df[register_col], errors="coerce")

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    filtered = df[(df["ip.src"].astype(str) == src_ip) & (reg_num == float(register_value))].copy()
    window_df = filtered[(filtered["aligned_ts"] >= start_ts) & (filtered["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")[regval_col].std().reindex(bins)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel(f"std({regval_col})")
    plt.title(
        f"Std register value for src={src_ip}, register={register_value} around {center_ts} (+/-{interval_seconds}s)"
    )
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


def plot_sum_write_fc_packets_per_src_ip(
    input_csv,
    src_ip,
    center_timestamp,
    interval_seconds,
    func_col="modbus.func_code",
    write_func_codes=(5, 6, 15, 16),
    time_offset_seconds=TIME_OFFSET,
):
    """
    3) Plot sum(write_fc_packets) per src IP in 1-second bins.

    A write_fc_packet is a packet with modbus.func_code in write_func_codes.
    """
    ipaddress.ip_address(src_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "frame.time_epoch", func_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(epoch[epoch.notna()] + float(time_offset_seconds), unit="s", errors="coerce")

    func_num = pd.to_numeric(df[func_col], errors="coerce")
    write_codes = {int(c) for c in write_func_codes}
    df["is_write_fc_packet"] = func_num.isin(write_codes).astype(int)

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    filtered = df[df["ip.src"].astype(str) == src_ip].copy()
    window_df = filtered[(filtered["aligned_ts"] >= start_ts) & (filtered["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")["is_write_fc_packet"].sum().reindex(bins, fill_value=0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel("sum(write_fc_packets)")
    plt.title(f"Write FC packet sum for src={src_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


def plot_nunique_src_ips_issuing_writes(
    input_csv,
    center_timestamp,
    interval_seconds,
    func_col="modbus.func_code",
    write_func_codes=(5, 6, 15, 16),
    time_offset_seconds=TIME_OFFSET,
):
    """
    4) Plot nunique(ip.src) issuing writes network-wide in 1-second bins.
    """
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "frame.time_epoch", func_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(epoch[epoch.notna()] + float(time_offset_seconds), unit="s", errors="coerce")

    func_num = pd.to_numeric(df[func_col], errors="coerce")
    write_codes = {int(c) for c in write_func_codes}
    df = df[func_num.isin(write_codes)].copy()

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    window_df = df[(df["aligned_ts"] >= start_ts) & (df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")["ip.src"].nunique().reindex(bins, fill_value=0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel("nunique(ip.src) issuing writes")
    plt.title(f"Unique write-issuing src IPs around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


# Example usage:
# input_csv = "../train/cscada_attack_ssw.csv"
# src_ip = "185.175.0.5"
# register_value = 40001
# center_timestamp = "2023-03-19 03:01:57.813"
# x = 20
#
# plot_mean_regval_per_src_register(input_csv, src_ip, register_value, center_timestamp, x)
# plot_std_regval_per_src_register(input_csv, src_ip, register_value, center_timestamp, x)
# plot_sum_write_fc_packets_per_src_ip(input_csv, src_ip, center_timestamp, x)
# plot_nunique_src_ips_issuing_writes(input_csv, center_timestamp, x)